# 04 · Instruction-Tuning Llama 2 (QLoRA + TRL)

End-to-end instruction tuning of **Llama-2-7b** with QLoRA, the workflow from the source repo, cleaned up and documented.
**Requires a CUDA GPU** and **Llama 2 access** (accept the license on Hugging Face and `huggingface-cli login`).
Data guidance: [docs/06_data_preparation.md](../docs/06_data_preparation.md).


## 1. Install + authenticate
Llama 2 is gated: request access at huggingface.co/meta-llama and log in.


In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes
# from huggingface_hub import notebook_login; notebook_login()


## 2. Config in one place


In [ ]:
import torch
model_name = 'meta-llama/Llama-2-7b-hf'
dataset_name = 'mlabonne/guanaco-llama2-1k'  # ready-made instruction dataset
new_model = 'llama-2-7b-finetuned'


## 3. Load dataset
`guanaco-llama2-1k` is already formatted with `<s>[INST] ... [/INST] ... </s>`, the Llama 2 chat template.


In [ ]:
from datasets import load_dataset
dataset = load_dataset(dataset_name, split='train')
print(len(dataset)); print(dataset[0]['text'][:300])


## 4. 4-bit base model + tokenizer


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map='auto')
model.config.use_cache = False; model.config.pretraining_tp = 1
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token; tokenizer.padding_side = 'right'


## 5. LoRA + training (SFTTrainer)


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import TrainingArguments
from trl import SFTTrainer

model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(r=64, lora_alpha=16, lora_dropout=0.1, bias='none',
    task_type='CAUSAL_LM')
args = TrainingArguments(output_dir='out_llama2', num_train_epochs=1,
    per_device_train_batch_size=4, gradient_accumulation_steps=2, learning_rate=2e-4,
    lr_scheduler_type='cosine', warmup_ratio=0.03, logging_steps=10,
    bf16=True, optim='paged_adamw_32bit', gradient_checkpointing=True, report_to='none')
trainer = SFTTrainer(model=model, args=args, train_dataset=dataset, peft_config=peft_config,
    dataset_text_field='text', max_seq_length=512, tokenizer=tokenizer)
trainer.train()
trainer.save_model(new_model)


## 6. Test the fine-tuned model


In [ ]:
from transformers import pipeline
prompt = '<s>[INST] What is a large language model? [/INST]'
pipe = pipeline('text-generation', model=trainer.model, tokenizer=tokenizer, max_new_tokens=120)
print(pipe(prompt)[0]['generated_text'])


## 7. Where to go next
- **Evaluate**: [docs/07_evaluation.md](../docs/07_evaluation.md).
- **Merge**: `python src/merge_adapter.py --base meta-llama/Llama-2-7b-hf --adapter llama-2-7b-finetuned --out merged_model`.
- **Deploy**: [docs/08_deployment.md](../docs/08_deployment.md) — FastAPI, HF Spaces, or Ollama.
- **Publish**: `python src/push_to_hub.py --path llama-2-7b-finetuned --repo <you>/llama-2-7b-finetuned`.
